# 2026 Roster Pipeline Audit

## tl;dr

This notebook verifies the player-level grain, current-roster filtering, historical
projection coverage, Week 1 opponent coverage, and auction-value validity used by
the Streamlit dashboard. Run the notebook to refresh the observed counts.

## Context & Methods

The application joins nflverse 2026 season rosters to 2023–2025 regular-season
player statistics using `gsis_id`. The intended grain is one row per rostered
player, plus one row per NFL D/ST.

### Key Assumptions

- `ACT` and `RES` are retained; `CUT` and `RET` are excluded.
- Players without historical NFL production remain visible but receive no
  projection and a $1 placeholder.
- Week 1 opponents come from the nflverse schedules release.
- The current market benchmark is an internal last-season baseline, not ADP.

## Data

### 1. Load the generated application tables

In [1]:
from pathlib import Path
import pandas as pd

root = Path.cwd()
if not (root / "data").exists():
    root = root.parent

board = pd.read_csv(root / "data" / "processed" / "draft_board_2026_current.csv")
recent = pd.read_csv(root / "data" / "processed" / "recent_games_2026.csv")
board.shape, recent.shape

((933, 32), (2048, 11))

## Results

### 2. Confirm grain, coverage, and valid domains

In [2]:
offense = board.loc[board["position"].ne("DST")]
audit = pd.Series({
    "board_rows": len(board),
    "duplicate_player_ids": int(board["player_id"].duplicated().sum()),
    "offensive_roster_rows": len(offense),
    "projected_offensive_players": int(offense["projected_points"].notna().sum()),
    "projection_coverage_rate": offense["projected_points"].notna().mean(),
    "week_1_opponent_coverage_rate": offense["opponent"].ne("TBD").mean(),
    "invalid_positions": int((~board["position"].isin(["QB","RB","WR","TE","DST"])).sum()),
    "auction_values_below_one": int(board["model_value"].lt(1).sum()),
})
audit.to_frame("value")

,value
board_rows,933.000000
duplicate_player_ids,0.000000
offensive_roster_rows,901.000000
projected_offensive_players,443.000000
projection_coverage_rate,0.491676
week_1_opponent_coverage_rate,1.000000
invalid_positions,0.000000
auction_values_below_one,0.000000


### 3. Inspect coverage by position

In [3]:
coverage = (
    offense.groupby("position")
    .agg(
        rostered_players=("player_id", "size"),
        projected_players=("projected_points", "count"),
        week_1_opponents=("opponent", lambda values: values.ne("TBD").sum()),
    )
)
coverage["projection_coverage"] = (
    coverage["projected_players"] / coverage["rostered_players"]
)
coverage

,rostered_players,projected_players,week_1_opponents,projection_coverage
position,,,,
QB,115,71,115,0.617391
RB,210,97,210,0.461905
TE,193,102,193,0.528497
WR,383,173,383,0.451697


### 4. Run decision-critical assertions

In [4]:
assert board["player_id"].is_unique
assert set(board["position"]) == {"QB", "RB", "WR", "TE", "DST"}
assert board["model_value"].ge(1).all()
assert offense["roster_status"].isin(["ACT", "RES"]).all()
assert offense["opponent"].ne("TBD").all()
assert board.loc[board["position"].eq("DST"), "model_value"].eq(1).all()
"All decision-critical checks passed."


'All decision-critical checks passed.'

## Takeaways

- The processed board is unique at the intended player/DST grain.
- Week 1 opponent coverage is complete for the current offensive roster pool.
- Historical projection coverage is intentionally incomplete: rookies and
  players without qualifying NFL production are preserved and labeled unavailable.
- Current teams and roster status are source-backed; projections and auction
  values remain model estimates.
- ADP, injuries, depth charts, and defensive strength are still required before
  treating auction edges or matchup labels as draft-ready recommendations.